In [13]:
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sbn
import numpy as np 

import torch

import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score
from sklearn.metrics import silhouette_score, silhouette_samples
import umap
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

from amlvae.models.VAE import VAE 
from amlvae.train.Trainer import Trainer

from amlvae.data.ExprProcessor import ExprProcessor
from amlvae.data.ClinProcessor import ClinProcessor

from sklearn.model_selection import KFold

from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split

import umap 
from sklearn.decomposition import PCA

# auto reimport 
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
train_df = pd.read_csv('./data/amlmds_train.csv', index_col=0)
val_df = pd.read_csv('./data/amlmds_val.csv', index_col=0)
test_df = pd.read_csv('./data/amlmds_test.csv', index_col=0)

In [15]:
x_train = torch.tensor(train_df.values, dtype=torch.float32)
x_val = torch.tensor(val_df.values, dtype=torch.float32)
x_test = torch.tensor(test_df.values, dtype=torch.float32)

print('# train samples:', x_train.shape[0])
print('# val samples:', x_val.shape[0])
print('# test samples:', x_test.shape[0]) 

print('# features:', x_train.shape[1])

# train samples: 1014
# val samples: 180
# test samples: 49
# features: 2042


In [16]:
model_kwargs = {
            'input_dim'   : x_train.size(1),
            'hidden_dim'  : 1024,
            'n_layers'    : 2,
            'latent_dim'  : 32,
            'norm'        : 'layer',
            'variational' : True,
            'dropout'     : 0.1,
            'nonlin'      : 'elu'
        }

lr = 5e-5 
l2 = 0
epochs = 250
beta = 10.0
batch_size = 128

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = VAE(**model_kwargs).to(device)
optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)

In [17]:
x_train = x_train.to(device)
x_val = x_val.to(device)
x_test = x_test.to(device)

In [18]:
best_elbo = float('inf')
patience_count = 0 
best_model = None 

for epoch in range(epochs): 

    model.train()
    losses = []
    for ixs in torch.split(torch.randperm(len(x_train)), batch_size):

        optim.zero_grad()
        x = x_train[ixs].to(device)
        
        out = model(x)
        loss, mse, kld, lm = model.loss(x, beta=beta, **out)

        loss.backward()
        optim.step()
        losses.append(loss.item())
    losses = np.mean(losses) 

    model.eval() 
    with torch.no_grad():   
        xhat = model(x_val)['xhat']
        val_r2  = r2_score(x_val.cpu().numpy(), xhat.cpu().numpy(), multioutput='uniform_average')
        val_mse = np.mean((x_val.cpu().numpy() - xhat.cpu().numpy())**2)

    print(f'Epoch {epoch+1}/{epochs}, Loss: {losses:.4f}, MSE: {mse:.4f}, KLD: {kld:.4f}, LM: {lm:.4f}, val R2: {val_r2:.3f}, val mse: {val_mse:.2f}', end='\r')

    
    

In [19]:
model.eval() 
with torch.no_grad():   
    xhat = model(x_test)['xhat']
    test_r2  = r2_score(x_test.cpu().numpy(), xhat.cpu().numpy(), multioutput='uniform_average')
    test_mse = np.mean((x_test.cpu().numpy() - xhat.cpu().numpy())**2)

print(f'\nTest R2: {test_r2:.3f}')
print(f'Test MSE: {test_mse:.3f}')


Test R2: 0.318
Test MSE: 0.987


In [20]:
X = np.concatenate([x_train.cpu().numpy(), x_val.cpu().numpy(), x_test.cpu().numpy()], axis=0)
ids = np.concatenate([train_df.index.values, val_df.index.values, test_df.index.values], axis=0)

In [21]:
model.eval()
with torch.no_grad(): 
    z, _ = model.encode(torch.tensor(X, dtype=torch.float32).to(device))

In [22]:
zdf = pd.DataFrame(z.cpu().numpy(), columns=[f'z{i}' for i in range(z.size(1))])
zdf = zdf.assign(id=ids)
zdf = zdf.assign(vae_partition = ['train'] * len(train_df) + ['val'] * len(val_df) + ['test'] * len(test_df))

zdf.to_csv('./data/amlmds_vae_zdf.csv', index=False)

In [23]:
zdf.head() 

,z0,z1,z2,z3,z4,z5,z6,z7,z8,z9,...,z24,z25,z26,z27,z28,z29,z30,z31,id,vae_partition
0,-0.269626,0.987643,-0.324002,0.546046,-0.375148,-0.066841,-1.068665,-0.600907,-1.152909,-1.997957,...,-1.102218,-0.741421,1.508016,-0.801014,0.419064,0.292559,-0.577267,0.650693,001454b2-aff9-4659-85a6-73fb8092589a,train
1,0.459493,-0.657565,1.781547,0.572654,0.414747,-1.502082,-0.527756,-1.495341,0.758157,-0.858126,...,0.467219,-1.096497,0.563714,-1.013973,0.666943,-0.928289,-0.133709,-0.488331,002cacd9-c03b-4526-a380-0701f41c4a9e,train
2,-0.365379,-0.430586,0.897652,-1.123011,0.318585,0.410173,-1.471870,-0.044682,-0.139588,-1.039250,...,0.759534,0.080889,1.098339,1.856437,-0.387810,0.719957,-1.070217,-0.371302,006e5777-2603-4db7-a1d6-8c8085c5e3e5,train
3,-0.203828,0.035070,-0.947490,-0.139151,-0.261438,0.144500,0.229609,0.991641,1.750866,0.682274,...,0.962986,-0.295481,-0.590958,-1.664098,-0.586027,-0.302337,0.551441,0.804774,00870f33-cab3-4c23-bd0d-8903a5a9789e,train
4,-0.486196,-0.993290,-2.102340,0.299851,-0.909638,-2.409446,0.171750,-0.036894,0.112995,0.442299,...,1.915407,-0.059691,-0.311332,-0.803625,0.713860,0.881199,-0.280586,-0.965163,00b535f6-064a-4dcf-ab14-387a54eedeee,train


In [24]:
torch.save(model, './data/amlmds_vae_model.pt')